# Data Acquisition and Preprocessing

## Project Overview

This project investigates gene expression differences between the Activated B-cell-like (ABC) and Germinal Center B-cell-like (GCB) subtypes of Diffuse Large B-cell Lymphoma (DLBCL).

The objective of this notebook is to:

1. Download the gene expression dataset and clinical metadata.
2. Map Affymetrix probe IDs to gene symbols.
3. Aggregate probe-level measurements into gene-level expression values.
4. Clean and curate clinical metadata.
5. Select patients treated with R-CHOP-like therapy.
6. Generate expression and metadata files for downstream analyses.

The processed data generated here will be used for PCA, differential expression analysis, pathway enrichment, and survival analysis.

## Biological Background

Diffuse Large B-cell Lymphoma (DLBCL) is the most common subtype of non-Hodgkin lymphoma.

Gene expression profiling has revealed that DLBCL consists of biologically distinct molecular subtypes:

- **GCB (Germinal Center B-cell-like)**: originates from germinal center B cells and generally has a more favorable prognosis.
- **ABC (Activated B-cell-like)**: resembles activated B cells and is associated with chronic NF-κB signaling and poorer clinical outcomes.

Understanding transcriptional differences between these subtypes can reveal biomarkers, dysregulated pathways, and potential therapeutic targets.

In [1]:
import pandas as pd
import numpy as np
import GEOparse

## Dataset Selection

We used the GEO dataset **GSE10846**, one of the largest publicly available DLBCL gene expression cohorts.

Dataset characteristics:

- Disease: Diffuse Large B-cell Lymphoma (DLBCL)
- Platform: Affymetrix Human Genome U133 Plus 2.0 Array (GPL570)
- Includes gene expression profiles and clinical annotations
- Contains molecular subtype classifications (ABC, GCB, and Unclassified)
- Includes survival information and treatment data

The corresponding platform annotation file (GPL570) was downloaded to map probe identifiers to gene symbols.

In [2]:
gse = GEOparse.get_GEO(geo="GSE10846", destdir="../data/raw")
gpl = GEOparse.get_GEO(geo="GPL570", destdir="../data/raw")

expr = gse.pivot_samples('VALUE')
metadata = gse.phenotype_data

14-Jun-2026 16:58:49 DEBUG utils - Directory ../data/raw already exists. Skipping.
14-Jun-2026 16:58:49 INFO GEOparse - File already exist: using local version.
14-Jun-2026 16:58:49 INFO GEOparse - Parsing ../data/raw\GSE10846_family.soft.gz: 
14-Jun-2026 16:58:49 DEBUG GEOparse - DATABASE: GeoMiame
14-Jun-2026 16:58:49 DEBUG GEOparse - SERIES: GSE10846
14-Jun-2026 16:58:49 DEBUG GEOparse - PLATFORM: GPL570
C:\Users\Nonye\anaconda3\envs\rnaseq_env\lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274895
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274896
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274897
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274898
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274899
14-Jun-2026 16:58:51 DEBUG GEOparse - SAMPLE: GSM274900
14-Jun-2

In [3]:
mapping = (
    gpl.table[["ID", "Gene Symbol"]]
    .dropna(subset=["Gene Symbol"])
    .drop_duplicates(subset=["ID"])
    .rename(columns={"ID": "probe", "Gene Symbol": "gene"})
)
mapping["gene"] = mapping["gene"].apply(lambda x: x.split(" /// ")[0] if isinstance(x, str) else x)

In [4]:
expr = expr.reset_index().rename(columns={"ID_REF": "probe"})
expr_annotated = expr.merge(mapping, on="probe", how="inner")

In [5]:
expr_annotated.head()

,probe,GSM274895,GSM274896,GSM274897,GSM274898,GSM274899,GSM274900,GSM274901,GSM274902,GSM274903,...,GSM275306,GSM275307,GSM275308,GSM361239,GSM361240,GSM361241,GSM361242,GSM361243,GSM361244,gene
0,1007_s_at,8.647,9.420,8.724,8.505,9.411,10.009,8.007,8.214,10.887,...,11.041,8.766,8.135,8.258,6.703,7.951,7.704,8.133,8.978,DDR1
1,1053_at,8.602,9.675,8.783,9.700,10.127,10.032,9.502,9.565,10.005,...,10.531,9.146,10.203,7.971,9.154,9.098,9.667,9.341,8.330,RFC2
2,117_at,8.207,8.442,7.799,6.758,6.977,7.510,10.705,6.698,7.401,...,8.241,6.869,7.339,8.952,6.162,11.449,9.392,11.593,10.691,HSPA6
3,121_at,10.295,10.658,10.650,9.976,9.820,10.001,9.941,10.176,9.809,...,9.143,9.835,8.911,8.846,9.150,8.908,8.556,8.334,9.017,PAX8
4,1255_g_at,5.555,6.739,5.878,5.681,6.616,5.661,3.217,5.689,8.907,...,6.394,5.621,6.638,7.312,6.779,3.511,3.087,3.121,4.700,GUCA1A


## Extracting Expression Data

The GEO series matrix was converted into a sample-by-expression matrix.

At this stage:

- Rows represent microarray probes.
- Columns represent patient samples.
- Expression values are already normalized by the original study authors.

## Mapping Probes to Genes

Microarray platforms measure expression using probe sets rather than gene symbols directly.

To improve biological interpretability:

1. Probe IDs were matched to gene symbols using the GPL570 annotation file.
2. Probes lacking gene annotations were removed.
3. Duplicate probe mappings were handled during aggregation.

This step converts probe-level measurements into gene-level expression estimates.

In [6]:
expr_gene = (
    expr_annotated
    .drop(columns=["probe"])
    .groupby("gene")
    .mean()  
)

In [7]:
expr_gene.shape

(22880, 420)

## Probe Aggregation

Multiple probes can map to the same gene.

To obtain a single expression value per gene, probe measurements mapping to the same gene symbol were averaged.

### Result

- Initial samples: 420
- Final genes: 22,880

The resulting matrix contains gene-level expression measurements suitable for downstream biological analyses.

## Clinical Metadata Processing

The GEO metadata contains demographic, clinical, treatment, and survival information.

Several cleaning steps were required:

- Standardize missing values.
- Remove samples lacking essential diagnostic information.
- Extract clinically relevant variables.
- Convert text-based annotations into analysis-ready variables.

In [8]:
metadata = metadata.replace(["NA", "N/A", "na", "", "Submitting diagnosis: NA"], np.nan)

In [9]:
metadata.isna().sum()

title                                    0
geo_accession                            0
status                                   0
submission_date                          0
last_update_date                         0
type                                     0
channel_count                            0
source_name_ch1                          0
organism_ch1                             0
taxid_ch1                                0
characteristics_ch1.0.Gender            24
characteristics_ch1.1.Age                6
characteristics_ch1.2.Tissue             0
characteristics_ch1.3.Disease state      3
characteristics_ch1.4.Individual         6
characteristics_ch1.5.Clinical info      6
characteristics_ch1.6.Clinical info      0
characteristics_ch1.7.Clinical info      0
characteristics_ch1.8.Clinical info      0
characteristics_ch1.9.Clinical info      0
characteristics_ch1.10.Clinical info     0
characteristics_ch1.11.Clinical info     0
characteristics_ch1.12.Clinical info     0
characteris

In [10]:
metadata = metadata.dropna(subset="characteristics_ch1.5.Clinical info") #remove samples with unknown diagnosis

## Clinical Variables Selected

The following variables were retained for downstream analyses:

| Variable | Description |
|-----------|-------------|
| Gender | Patient sex |
| Age | Age at diagnosis |
| Subtype | ABC or GCB molecular subtype |
| Status | Alive or Dead at follow-up |
| Followup_years | Follow-up duration |
| Chemotherapy | Treatment regimen |
| Stage | Clinical stage |

In [11]:
clean_metadata = pd.DataFrame({"Gender": metadata['characteristics_ch1.0.Gender'],
                              "Age": metadata['characteristics_ch1.1.Age'],
                              "Subtype": metadata['characteristics_ch1.6.Clinical info'],
                              "Status": metadata['characteristics_ch1.7.Clinical info'],
                              "Followup_years": metadata['characteristics_ch1.8.Clinical info'],
                              "Chemotherapy": metadata['characteristics_ch1.9.Clinical info'],
                              "Stage": metadata['characteristics_ch1.11.Clinical info']})  #select useful columns from metadata

In [12]:
clean_metadata

,Gender,Age,Subtype,Status,Followup_years,Chemotherapy,Stage
GSM274895,male,52,Final microarray diagnosis: GCB DLBCL,Follow up status: DEAD,Follow up years: 2.68,Chemotherapy: CHOP-Like Regimen,Stage: 3
GSM274896,male,75,Final microarray diagnosis: GCB DLBCL,Follow up status: DEAD,Follow up years: 0.82,Chemotherapy: CHOP-Like Regimen,Stage: 3
GSM274897,female,76,Final microarray diagnosis: ABC DLBCL,Follow up status: DEAD,Follow up years: 2.54,Chemotherapy: CHOP-Like Regimen,Stage: 1
GSM274898,female,56,Final microarray diagnosis: ABC DLBCL,Follow up status: ALIVE,Follow up years: 9.67,Chemotherapy: CHOP-Like Regimen,Stage: 3
GSM274899,male,46,Final microarray diagnosis: ABC DLBCL,Follow up status: ALIVE,Follow up years: 4.83,Chemotherapy: CHOP-Like Regimen,Stage: 2
...,...,...,...,...,...,...,...
GSM275304,male,56,Final microarray diagnosis: ABC DLBCL,Follow up status: DEAD,Follow up years: 0.46,Chemotherapy: R-CHOP-Like Regimen,Stage: 4
GSM275305,female,42,Final microarray diagnosis: ABC DLBCL,Follow up status: DEAD,Follow up years: 1.21,Chemotherapy: R-CHOP-Like Regimen,Stage: 3
GSM275306,female,61,Final microarray diagnosis: GCB DLBCL,Follow up status: ALIVE,Follow up years: 0.47,Chemotherapy: R-CHOP-Like Regimen,Stage: 1
GSM275307,female,58,Final microarray diagnosis: ABC DLBCL,Follow up status: DEAD,Follow up years: 1.16,Chemotherapy: R-CHOP-Like Regimen,Stage: 4


In [13]:
clean_metadata["Stage"] = clean_metadata["Stage"].str.replace("Stage: ", "", regex=False).replace("NA", np.nan)
clean_metadata["Followup_years"] = clean_metadata["Followup_years"].str.replace("Follow up years: ", "", regex=False).astype(float)

In [14]:
clean_metadata["Subtype"] = clean_metadata["Subtype"].replace({"Final microarray diagnosis: GCB DLBCL": "GCB", "Final microarray diagnosis: ABC DLBCL": "ABC", "Final microarray diagnosis: Unclassified DLBCL" : "Unclassified"})
clean_metadata["Status"] = clean_metadata["Status"].replace({"Follow up status: ALIVE": "Alive", "Follow up status: DEAD": "Dead" })
clean_metadata["Chemotherapy"] = clean_metadata["Chemotherapy"].replace({"Chemotherapy: R-CHOP-Like Regimen" : "R-CHOP-Like", "Chemotherapy: CHOP-Like Regimen" : "CHOP-Like"})

In [15]:
clean_metadata["Age"] = clean_metadata["Age"].astype(float)

In [16]:
clean_metadata["Event_Status"] = clean_metadata["Status"].map({"Alive": 0, "Dead": 1})

## Cohort Selection

To create a biologically homogeneous cohort:

1. Unclassified DLBCL cases were removed.
2. Only patients treated with an R-CHOP-like regimen were retained.

### Rationale

The primary objective is to compare molecular differences between ABC and GCB DLBCL while minimizing variation introduced by:

- Ambiguous subtype classification.
- Differences in treatment strategy.

Restricting the analysis to R-CHOP-treated patients improves cohort consistency for both differential expression and survival analyses.

In [17]:
clean_metadata = clean_metadata[clean_metadata["Subtype"] != "Unclassified"] 

In [18]:
clean_metadata = clean_metadata[clean_metadata["Chemotherapy"] == "R-CHOP-Like"] 

In [19]:
clean_metadata

,Gender,Age,Subtype,Status,Followup_years,Chemotherapy,Stage,Event_Status
GSM275076,male,73.0,ABC,Alive,2.52,R-CHOP-Like,4,0
GSM275077,female,71.0,GCB,Alive,1.71,R-CHOP-Like,2,0
GSM275078,male,71.0,GCB,Alive,4.74,R-CHOP-Like,3,0
GSM275080,male,83.0,ABC,Alive,3.42,R-CHOP-Like,1,0
GSM275081,male,63.0,GCB,Alive,5.23,R-CHOP-Like,2,0
...,...,...,...,...,...,...,...,...
GSM275304,male,56.0,ABC,Dead,0.46,R-CHOP-Like,4,1
GSM275305,female,42.0,ABC,Dead,1.21,R-CHOP-Like,3,1
GSM275306,female,61.0,GCB,Alive,0.47,R-CHOP-Like,1,0
GSM275307,female,58.0,ABC,Dead,1.16,R-CHOP-Like,4,1


## Final Cohort

After quality control and filtering:

- Total patients retained: **200**
- Molecular subtypes:
  - ABC DLBCL
  - GCB DLBCL
- Treatment:
  - R-CHOP-like regimen only

These samples form the final analytical cohort used throughout the remainder of the project.

In [20]:
clean_metadata["Subtype"].value_counts()

Subtype
GCB    107
ABC     93
Name: count, dtype: int64

## Sample Alignment Verification

Before downstream analyses, the expression matrix and clinical metadata were checked to ensure identical sample ordering.

This verification is critical because mismatched sample labels can invalidate statistical analyses and biological conclusions.

In [21]:
library_columns = clean_metadata.index.tolist() 
expr_final = expr_gene[library_columns]

In [22]:
if list(expr_final.columns) == list(clean_metadata.index): 
    print("Sample alignment successful.") 
    print(f"expression matrix shape: {expr_final.shape}") 
    print(f"Metadata shape: {clean_metadata.shape}") 
else: 
    raise ValueError( 
        "Alignment Error: Count matrix columns " 
        "do not match metadata indices."
    )

Sample alignment successful.
expression matrix shape: (22880, 200)
Metadata shape: (200, 8)


In [23]:
expr_final.to_csv("../data/processed/GSE10846_expr_matrx.csv")
clean_metadata.to_csv("../data/processed/GSE10846_metadata.csv")

## Exporting Processed Data

The cleaned expression matrix and curated metadata were exported for use in subsequent notebooks:

1. Principal Component Analysis (PCA)
2. Differential Expression Analysis
3. Pathway Enrichment Analysis
4. Survival Analysis

### Output Files

- GSE10846_expr_matrix.csv
- GSE10846_metadata.csv